In [6]:
from Bio import SeqIO
import gzip
import re
import pandas as pd

In [12]:
# -------- Parameters --------

motif = "CGAACNNNNGTTCG"

max_mismatches = 1

upstream_start = -400
upstream_end = -10

# Maximum intergenic distance for operon prediction
max_gap = 100

# -------- Input files ---------
genome_fasta = "AM1001-oriented.fasta"
gtf_file = "annot.gff"

In [13]:
# -------- Helper Functions --------

def hamming_distance(s1, s2):
    return sum(a != b for a, b in zip(s1, s2))


def motif_match_with_N(motif, seq_window, max_mismatches=0):

    mismatches = 0

    for m, s in zip(motif, seq_window):

        if m == "N":
            continue  # N matches anything

        if m != s:

            mismatches += 1

            if mismatches > max_mismatches:
                return False, mismatches

    return True, mismatches


def reverse_complement(seq):

    complement = str.maketrans("ATCGN", "TAGCN")

    return seq.translate(complement)[::-1]


# -------- Load Genome --------

def load_genome(fasta_path):

    genome = {}

    opener = gzip.open if fasta_path.endswith(".gz") else open

    with opener(fasta_path, "rt") as handle:

        for record in SeqIO.parse(handle, "fasta"):

            genome[record.id] = str(record.seq).upper()

    return genome


# -------- Parse GFF (identify operons and get TSS) --------

def parse_gtf_tss(gtf_path, max_gap=100):

    genes = []

    opener = gzip.open if gtf_path.endswith(".gz") else open

    with opener(gtf_path, "rt") as f:

        for line in f:

            if line.startswith("#"):
                continue

            parts = line.strip().split("\t")

            if len(parts) < 9:
                continue

            if parts[2] != "gene":
                continue

            chrom = parts[0]

            start = int(parts[3])
            end = int(parts[4])

            strand = parts[6]

            if strand not in ["+", "-"]:
                continue

            # Parse GFF3 attributes
            attr_field = parts[8]

            attr_dict = dict(
                item.split("=", 1)
                for item in attr_field.strip().split(";")
                if "=" in item
            )

            # Get gene name
            gene_id = (
                attr_dict.get("Name")
                or attr_dict.get("gene")
                or attr_dict.get("ID")
                or "unknown"
            )

            genes.append({

                "chrom": chrom,
                "start": start,
                "end": end,
                "strand": strand,
                "gene_id": gene_id

            })

    # Sort genes by genomic position
    genes.sort(key=lambda x: (x["chrom"], x["start"]))

    # Identify possible operons
    operons = []

    current_group = []

    for gene in genes:

        if not current_group:

            current_group.append(gene)

            continue

        previous = current_group[-1]

        # Intergenic distance (GFF coordinates are 1-based)
        gap = gene["start"] - previous["end"] - 1

        same_contig = gene["chrom"] == previous["chrom"]

        same_strand = gene["strand"] == previous["strand"]

        if same_contig and same_strand and gap <= max_gap:

            current_group.append(gene)

        else:

            operons.append(current_group)

            current_group = [gene]

    if current_group:

        operons.append(current_group)

    # Define approximate TSS for each operon
    tss_list = []

    for i, group in enumerate(operons):

        strand = group[0]["strand"]

        if strand == "+":

            # First gene in transcriptional direction
            tss = group[0]["start"]

        else:

            # First gene in transcriptional direction
            tss = group[-1]["end"]

        gene_ids = [g["gene_id"] for g in group]

        operon_id = f"OP{i+1:04d}"

        tss_list.append((

            group[0]["chrom"],
            tss,
            strand,
            operon_id,
            gene_ids

        ))

    return tss_list


# -------- Scan for Motif --------

def find_motifs_near_tss(
    genome,
    tss_list,
    motif,
    max_mismatches,
    upstream_start,
    upstream_end
):

    motif_hits = []

    motif_len = len(motif)

    for chrom, tss, strand, operon_id, genes in tss_list:

        if chrom not in genome:
            continue

        seq = genome[chrom]

        # Extract upstream region
        # GFF coordinates: 1-based
        # Python sequence indices: 0-based

        if strand == "+":

            # Upstream region on positive strand

            region_start = max(
                0,
                tss - 1 + upstream_start
            )

            region_end = min(
                len(seq),
                tss + upstream_end
            )

            region = seq[region_start:region_end]

        else:

            # Upstream region on negative strand

            region_start = max(
                0,
                tss - upstream_end - 1
            )

            region_end = min(
                len(seq),
                tss - upstream_start
            )

            # Orient sequence in transcriptional direction
            region = reverse_complement(
                seq[region_start:region_end]
            )

        # Scan for motif
        for i in range(0, len(region) - motif_len + 1):

            window = region[i:i + motif_len]

            matched, mismatches = motif_match_with_N(
                motif,
                window,
                max_mismatches
            )

            if matched:

                # Position relative to TSS
                # Same convention for both strands

                relative_pos = upstream_start + i

                # Absolute genomic motif coordinates
                # motif_start = leftmost genomic coordinate (1-based)

                if strand == "+":

                    motif_start_abs = region_start + i + 1

                else:

                    motif_start_abs = (
                        region_end - i - motif_len + 1
                    )

                motif_end_abs = motif_start_abs + motif_len - 1

                motif_hits.append({

                    "operon_id": operon_id,

                    "genes_in_operon": ",".join(genes),

                    "n_genes": len(genes),

                    "chrom": chrom,

                    "tss": tss,

                    "strand": strand,

                    "motif_start": motif_start_abs,

                    "motif_end": motif_end_abs,

                    "rel_position_motif": relative_pos,

                    "mismatches": mismatches,

                    "matched_seq": window

                })

    return motif_hits

In [14]:
# -------- Main Runner --------

def main():

    # Load genome
    print("Loading genome...")

    genome = load_genome(genome_fasta)

    # Identify possible operons
    print("Identifying possible operons...")

    tss_list = parse_gtf_tss(
        gtf_file,
        max_gap=max_gap
    )

    print(f"Possible operons identified: {len(tss_list)}")

    # Count operons containing multiple genes
    multi_gene_operons = [
        op for op in tss_list if len(op[4]) > 1
    ]

    print(
        f"Operons containing multiple genes: "
        f"{len(multi_gene_operons)}"
    )

    print("\nOperons containing multiple genes:")

    for op in multi_gene_operons:

        print(
            op[3],
            op[4],
            "strand:", op[2],
            "TSS:", op[1]
        )

    # Search for motifs
    print("\nSearching for motifs...")

    hits = find_motifs_near_tss(
        genome,
        tss_list,
        motif,
        max_mismatches,
        upstream_start,
        upstream_end
    )

    print(f"\nMotif hits found: {len(hits)}")

    # Save results
    df = pd.DataFrame(hits)

    df.to_csv("motif_hits.csv", index=False)

    print("\nSaved results to motif_hits.csv")

    # Summary
    if not df.empty:

        print("\nSummary by strand:")

        print(df["strand"].value_counts())

        print("\nSummary by number of genes in operon:")

        print(df["n_genes"].value_counts().sort_index())

        print("\nRelative motif positions:")

        print(df["rel_position_motif"].describe())


if __name__ == "__main__":

    main()


Loading genome...
Identifying possible operons...
Possible operons identified: 1970
Operons containing multiple genes: 775

Operons containing multiple genes:
OP0003 ['rlbA', 'recF', 'remB', 'gyrB'] strand: + TSS: 2797
OP0005 ['pgaptmp_000008', 'pgaptmp_000009', 'pgaptmp_000010', 'pgaptmp_000011', 'rrf'] strand: + TSS: 9415
OP0009 ['pdxS', 'pdxT'] strand: + TSS: 18673
OP0013 ['dck', 'pgaptmp_000022', 'pgaptmp_000023', 'pgaptmp_000024'] strand: - TSS: 26638
OP0014 ['tadA', 'ffs'] strand: + TSS: 26721
OP0015 ['dnaX', 'pgaptmp_000028', 'recR', 'pgaptmp_000030', 'pgaptmp_000031'] strand: + TSS: 27686
OP0016 ['pgaptmp_000032', 'pgaptmp_000033', 'pgaptmp_000034', 'pgaptmp_000035', 'rrf'] strand: + TSS: 31218
OP0018 ['pgaptmp_000038', 'pgaptmp_000039', 'pgaptmp_000040', 'tmk', 'darA', 'pgaptmp_000043', 'holB', 'ricT', 'yabA', 'pgaptmp_000047', 'pgaptmp_000048', 'rsmI'] strand: + TSS: 36730
OP0020 ['metG', 'pgaptmp_000052'] strand: + TSS: 46452
OP0022 ['rnmV', 'rsmA'] strand: + TSS: 50881
OP00

In [15]:
tss_list = parse_gtf_tss(gtf_file, max_gap=100)

for op in tss_list[:10]:
    print(op)

('tig00000001_polypolish', 1, '+', 'OP0001', ['dnaA'])
('tig00000001_polypolish', 1526, '+', 'OP0002', ['dnaN'])
('tig00000001_polypolish', 2797, '+', 'OP0003', ['rlbA', 'recF', 'remB', 'gyrB'])
('tig00000001_polypolish', 6593, '+', 'OP0004', ['gyrA'])
('tig00000001_polypolish', 9415, '+', 'OP0005', ['pgaptmp_000008', 'pgaptmp_000009', 'pgaptmp_000010', 'pgaptmp_000011', 'rrf'])
('tig00000001_polypolish', 15404, '-', 'OP0006', ['pgaptmp_000013'])
('tig00000001_polypolish', 15525, '+', 'OP0007', ['guaB'])
('tig00000001_polypolish', 17144, '+', 'OP0008', ['dacA'])
('tig00000001_polypolish', 18673, '+', 'OP0009', ['pdxS', 'pdxT'])
('tig00000001_polypolish', 20488, '+', 'OP0010', ['serS'])


In [17]:
genome_fasta = "AM1001-oriented.fasta"

genome = load_genome(genome_fasta)

print(f"Contigs carregados: {len(genome)}")
print(f"Contig: {list(genome.keys())[0]}")
print(f"Tamanho: {len(list(genome.values())[0])} bp")

Contigs carregados: 1
Contig: tig00000001_polypolish
Tamanho: 3976591 bp


In [18]:
chrom, tss, strand, operon_id, genes = tss_list[0]

seq = genome[chrom]

region_start = max(0, tss - 1 - 400)
region_end = min(len(seq), tss + 10)

region = seq[region_start:region_end]

print("Operon:", operon_id)
print("Genes:", genes)
print("TSS:", tss)
print("Strand:", strand)
print("Region start:", region_start)
print("Region end:", region_end)
print("Region length:", len(region))
print("Region sequence:", region)

Operon: OP0001
Genes: ['dnaA']
TSS: 1
Strand: +
Region start: 0
Region end: 11
Region length: 11
Region sequence: ATGGAGAATAT


In [19]:
import inspect

print(inspect.getsource(find_motifs_near_tss))

def find_motifs_near_tss(
    genome,
    tss_list,
    motif,
    max_mismatches,
    upstream_start,
    upstream_end
):

    motif_hits = []

    motif_len = len(motif)

    for chrom, tss, strand, operon_id, genes in tss_list:

        if chrom not in genome:
            continue

        seq = genome[chrom]

        # Extract upstream region
        # GFF coordinates: 1-based
        # Python sequence indices: 0-based

        if strand == "+":

            # Upstream region on positive strand

            region_start = max(
                0,
                tss - 1 + upstream_start
            )

            region_end = min(
                len(seq),
                tss + upstream_end
            )

            region = seq[region_start:region_end]

        else:

            # Upstream region on negative strand

            region_start = max(
                0,
                tss - upstream_end - 1
            )

            region_end = min(
                len(seq),
     

In [20]:
hits = find_motifs_near_tss(
    genome,
    tss_list,
    motif,
    max_mismatches,
    -400,
    10
)

df = pd.DataFrame(hits)

print("Total de hits:", len(df))

print("\nHits atribuídos ao dnaA:")

print(
    df[df["genes_in_operon"] == "dnaA"].to_string(index=False)
)

Total de hits: 18

Hits atribuídos ao dnaA:
Empty DataFrame
Columns: [operon_id, genes_in_operon, n_genes, chrom, tss, strand, motif_start, motif_end, rel_position_motif, mismatches, matched_seq]
Index: []


In [21]:
df = pd.DataFrame(hits)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

print(
    df[
        [
            "operon_id",
            "genes_in_operon",
            "n_genes",
            "strand",
            "tss",
            "rel_position_motif",
            "mismatches",
            "matched_seq"
        ]
    ].to_string(index=False)
)

operon_id                    genes_in_operon  n_genes strand     tss  rel_position_motif  mismatches    matched_seq
   OP0057                     pgaptmp_000183        1      -  156772                -395           1 CGAACGCGCGTTTG
   OP0139      pgaptmp_000328,pgaptmp_000329        2      +  302003                -309           1 CGTACCACAGTTCG
   OP0157      pgaptmp_000358,pgaptmp_000359        2      +  334824                -309           1 CGTACCACAGTTCG
   OP0306      pgaptmp_000620,pgaptmp_000621        2      +  589055                -259           1 CGCACTGGAGTTCG
   OP0424      pgaptmp_000836,pgaptmp_000837        2      +  823566                -309           1 CGAACCGGAATTCG
   OP0491                     pgaptmp_000965        1      +  936367                  -5           1 CGAACATGAGTTGG
   OP0664                     pgaptmp_001262        1      - 1210101                 -29           1 AGAACATATGTTCG
   OP0862                               recA        1      + 1690608    

In [22]:
print("Menor posição:", df["rel_position_motif"].min())
print("Maior posição:", df["rel_position_motif"].max())

print("\nResultados fora da janela:")

print(
    df[
        (df["rel_position_motif"] < -400) |
        (df["rel_position_motif"] > 10)
    ]
)

Menor posição: -395
Maior posição: -5

Resultados fora da janela:
Empty DataFrame
Columns: [operon_id, genes_in_operon, n_genes, chrom, tss, strand, motif_start, motif_end, rel_position_motif, mismatches, matched_seq]
Index: []
